# Explore `unmatched_market_rows.csv`

Goal: understand why ~13k tennis-data.co.uk rows fail to match a row in our `matches` table even though both player names resolved.

Four candidate causes (from `data/processed/unmatched_market_rows.csv`):
1. AliasIndex resolved to the wrong same-surname player.
2. Date offset > our ±10-day window.
3. Tournament present in tennis-data but absent from Sackmann.
4. Winner/loser swap between sources.

This notebook quantifies which one dominates.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

DATA_DIR = Path("../data/processed")
CSV_PATH = DATA_DIR / "unmatched_market_rows.csv"
DUCKDB_PATH = DATA_DIR / "tennis.duckdb"

df = pd.read_csv(CSV_PATH, parse_dates=["date"])
df["year"] = df["date"].dt.year
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print()
print("Reason distribution:")
print(df["reason"].value_counts())
df.head(3)

## 1. Distribution by tour and year

If unmatched rows cluster on specific years, the cause is probably an *external* problem (e.g., schema change in tennis-data, missing tournaments in Sackmann).

If they're roughly proportional to total matches per year, the cause is *internal* (reconciliation, date tolerance).

In [ ]:
pivot = df.groupby(["tour", "year"]).size().unstack("tour", fill_value=0)
pivot["TOTAL"] = pivot.sum(axis=1)
pivot

## 2. Who are the biggest "losers"?

Top raw-name + matched-id pairs by frequency. A single name that fails 100+ times is almost certainly a systematic reconciliation bug — easy win.

In [ ]:
stacked = pd.concat(
    [
        df[["tour", "winner_raw", "winner_player_id"]].rename(
            columns={"winner_raw": "raw", "winner_player_id": "matched_id"}
        ),
        df[["tour", "loser_raw", "loser_player_id"]].rename(
            columns={"loser_raw": "raw", "loser_player_id": "matched_id"}
        ),
    ],
    ignore_index=True,
)

top = (
    stacked.groupby(["raw", "matched_id", "tour"])
    .size()
    .reset_index(name="unmatched_count")
    .sort_values("unmatched_count", ascending=False)
    .head(30)
)

conn = duckdb.connect(str(DUCKDB_PATH), read_only=True)
players = conn.execute(
    "SELECT player_id AS matched_id, full_name AS matched_full_name, ioc, dob FROM players"
).fetchdf()

top_annotated = top.merge(players, on="matched_id", how="left")
top_annotated

**Reading this table:** for each row, ask yourself: does `raw` look like it could be `matched_full_name`?

- ✅ `raw="Federer R."` matched to `Roger Federer` — match looks correct, must be cause #2/3/4 (date or tournament gap).
- ❌ `raw="Murray A."` matched to `Aleksandar Murray` (random other player) — clearly the wrong player. Cause #1.

The first column tells you the *symptom* count. The mismatches between `raw` and `matched_full_name` tell you the *cause*.

## 3. Investigate a specific suspicious case

Pick a row from the table above where `raw` clearly doesn't match `matched_full_name` and look for the right canonical player.

Edit `SUSPECT_RAW` and `SUSPECT_SURNAME` to whatever you want to investigate.

In [ ]:
SUSPECT_RAW = "Murray A."
SUSPECT_SURNAME = "Murray"
SUSPECT_TOUR = "ATP"

print(f"Sample unmatched rows for '{SUSPECT_RAW}':")
sample = df[
    (df["winner_raw"] == SUSPECT_RAW) | (df["loser_raw"] == SUSPECT_RAW)
][["date", "tour", "winner_raw", "winner_player_id", "loser_raw", "loser_player_id"]].head(5)
display(sample)

print(f"\nWhat AliasIndex currently picks for '{SUSPECT_RAW}':")
from tennis_predictor.data import reconcile  # noqa: E402

idx = reconcile.AliasIndex(conn, SUSPECT_TOUR)
result = idx.lookup(SUSPECT_RAW)
print(f"  status={result.status}  confidence={result.confidence:.3f}")
print(f"  matched={result.candidate_name!r}  canonical_id={result.canonical_player_id}")

print(f"\nAll {SUSPECT_TOUR} players with surname '{SUSPECT_SURNAME}':")
candidates = conn.execute(
    "SELECT player_id, full_name, dob, ioc "
    "FROM players WHERE tour = ? AND name_last ILIKE ? "
    "ORDER BY full_name",
    [SUSPECT_TOUR, SUSPECT_SURNAME],
).fetchdf()
candidates

## 4. Quantify cause #2 — date offset out of window

For each unmatched row, look in the matches table for the same (winner_id, loser_id) pair — regardless of date — and check how far the closest match is from the tennis-data date. If many cases have a close match just outside the ±10 day window, we should widen it.

In [ ]:
# Take a sample (~500 rows) so the per-row query stays cheap
sample_df = df.sample(min(500, len(df)), random_state=42)

offsets = []
for _, row in sample_df.iterrows():
    matches = conn.execute(
        "SELECT tourney_date FROM matches "
        "WHERE tour = ? AND winner_player_id = ? AND loser_player_id = ?",
        [row["tour"], row["winner_player_id"], row["loser_player_id"]],
    ).fetchall()
    if matches:
        best = min(abs((pd.Timestamp(m[0]) - row["date"]).days) for m in matches)
        offsets.append(best)

offsets_series = pd.Series(offsets)
print(f"Sample size: {len(sample_df)} rows")
print(f"  Found a (winner, loser) match in DB at any date: {len(offsets_series):,}")
print(f"  Of those, days from tennis-data date to closest match:")
print(offsets_series.describe())
print()
print("Histogram (capped at 60 days):")
offsets_series.clip(upper=60).hist(bins=30)

**How to read this:**
- If most offsets are 0-30 days → widening `_DATE_TOLERANCE_DAYS` from 10 to 30 would recover many rows. Cheap win.
- If most offsets are huge (years) → the (winner, loser) pair in unmatched DOESN'T actually correspond to a Sackmann match. Most likely: AliasIndex picked the wrong player, or it's a tennis-data row with no Sackmann counterpart at all.
- If the "Found a match" count is much smaller than sample size → most unmatched rows have NO Sackmann match for the (winner, loser) pair, so cause is reconciliation (#1) or external (#3).

## 5. Cheap fix candidates: stable "raw → wrong canonical" mappings

Names that fail at least N times AND whose matched_full_name clearly differs from the raw form. If we override these in `player_aliases` with the correct canonical_id (similar to how `apply_aliases_review.py` works), most of the unmatched rows would recover.

Run this once you've identified the right canonical IDs (manually, using the cell above).

In [ ]:
# Build a candidate list: each unique (raw, matched_id) pair with >= MIN_COUNT failures
MIN_COUNT = 20

candidates = (
    stacked.groupby(["raw", "matched_id", "tour"])
    .size()
    .reset_index(name="unmatched_count")
)
candidates = candidates[candidates["unmatched_count"] >= MIN_COUNT]
candidates = candidates.merge(players, on="matched_id", how="left")

# Heuristic: is the raw form plausibly the same person as matched_full_name?
# A loose check: does last-name token of `raw` appear in matched_full_name?
def looks_off(raw: str, matched: str | None) -> bool:
    if not isinstance(matched, str):
        return True
    raw_last = raw.split()[0].rstrip(".,'").lower()
    return raw_last not in matched.lower()

candidates["surname_mismatch"] = candidates.apply(
    lambda r: looks_off(r["raw"], r["matched_full_name"]), axis=1
)

# Suspect rows (likely wrong reconciliation)
suspects = candidates[candidates["surname_mismatch"]].sort_values(
    "unmatched_count", ascending=False
)
print(f"Suspicious 'raw -> wrong canonical' pairs with >= {MIN_COUNT} failures:")
suspects.head(30)

## What to do with this

If a small number of `raw → wrong_id` pairs account for many rows in the table above, build a CSV that says `raw → correct_player_id` for each and pipe through a thin script (mirror of `apply_aliases_review.py`):

```python
INSERT INTO player_aliases (alias_text, tour, source, canonical_player_id, confidence)
VALUES (?, ?, 'manual_override', ?, 1.0)
ON CONFLICT DO NOTHING;
```

On the next `refresh_data.py --clean` (or just rerun of market loader), those names will resolve to the correct canonical_id and the matches will join successfully.

If most unmatched rows are diverse (long tail, no concentrated wins) — leave them. 75% market coverage is good enough for the calibration benchmark.